# Prepare prompts

Przygotowuje prompty do tłumaczeń na podstawie szablonu `prompt_template.txt` i transkrypcji TED `ted_talks_transcripts.tsv`. 

#### 1. Zależności

In [1]:
import pandas as pd

#### 2. Ustawienia

In [2]:
TRANSCRIPTS_PATH = "./data/ted_talks_transcripts.tsv"
TEMPLATE_PATH = "./prompt_template.txt"
OUTPUT_PATH = "./data/prompts.tsv"

TARGET_LANGUAGES = {
    "zh": "Chinese",
    "fi": "Finnish",
    "fr": "French",
    "he": "Hebrew",
    "ja": "Japanese",
    "pl": "Polish",
}

#### 3. Wczytaj transkrypcje i szablon promptów

In [3]:
transcripts = pd.read_csv(TRANSCRIPTS_PATH, sep="	")
template = open(TEMPLATE_PATH, encoding="utf-8").read()

print(f"Wczytano {len(transcripts)} talkow")
print(f"Template: {len(template)} znakow")

Wczytano 277 talkow
Template: 3194 znakow


#### 4. Przygotuj prompty

In [ ]:
NUM_EXAMPLE_CUES = 5 # Liczba cue'ów docelowego tłumaczenia; przykład dla modelu
NUM_TRANSCRIPT_CUES = 100 # Liczba cue'ów z oryginalnego angielskiego transkryptu


def first_n_cues(srt_text, n):
    """Zwraca pierwsze n napisów z tekstu SRT."""
    if not isinstance(srt_text, str) or not srt_text.strip():
        return None
    cues = srt_text.split("\n\n")
    return "\n\n".join(cues[:n])


def build_prompt(template, lang_name, transcript, style_example):
    return (template
            .replace("[LANGUAGE]", lang_name)
            .replace("[STYLE_EXAMPLE]", style_example)
            .replace("[TRANSCRIPT]", transcript))


rows = []
skipped = 0
for _, talk in transcripts.iterrows():
    en_text = first_n_cues(talk.get("en_timed"), NUM_TRANSCRIPT_CUES)
    if not en_text:
        continue
    slug = talk["slug"]
    for lang_code, lang_name in TARGET_LANGUAGES.items():
        target_srt = talk.get(f"{lang_code}_timed")
        style_example = first_n_cues(target_srt, NUM_EXAMPLE_CUES)
        if not style_example:
            skipped += 1
            continue
        rows.append({
            "id": f"{lang_code}--{slug}",
            "slug": slug,
            "target_lang": lang_code,
            "target_lang_name": lang_name,
            "prompt": build_prompt(template, lang_name, en_text, style_example),
        })

df = pd.DataFrame(rows)
print(f"Lacznie promptow: {len(df)}")
print(f"Pominieto (brak ground truth): {skipped}")
print(f"Per jezyk: {df['target_lang'].value_counts().to_dict()}")

Lacznie promptow: 1661
Pominieto (brak ground truth): 1
Per jezyk: {'zh': 277, 'fr': 277, 'ja': 277, 'he': 277, 'pl': 277, 'fi': 276}


#### 5. Zapis

In [5]:
df.to_csv(OUTPUT_PATH, sep="	", index=False)
print(f"Zapisano -> {OUTPUT_PATH}")

Zapisano -> ./data/prompts.tsv


#### 6. Wyświetlanie

Wstępna walidacja wyników

In [6]:
df.head(3)

,id,slug,target_lang,target_lang_name,prompt
0,zh--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,zh,Chinese,You are a professional translator. Translate t...
1,fi--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fi,Finnish,You are a professional translator. Translate t...
2,fr--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fr,French,You are a professional translator. Translate t...


In [7]:
print(df.iloc[0]["prompt"])

You are a professional translator. Translate the following English TED Talk
transcript into Chinese.

The transcript is in SRT subtitle format. Each cue block consists of:
  1. a cue number
  2. a timestamp line (HH:MM:SS,mmm --> HH:MM:SS,mmm)
  3. one or more lines of text

CRITICAL - completeness (read first):
- You MUST translate EVERY SINGLE cue, from the first to the very last one in the input.
- Your output MUST contain exactly as many cue blocks as the input. Do not stop early.
- NEVER skip, summarize, abbreviate, or collapse cues. Translate each one in full.
- NEVER write meta-comments such as "(the rest is kept as in the original)",
  "(remaining cues continue in the same manner)", "..." or any placeholder in place
  of a real translation. Every cue must contain the actual translated text.
- If the transcript is long, keep going regardless — finish all cues.

Rules:
- Preserve every cue number exactly as it appears
- Preserve every timestamp line exactly as it appears
- Transl